# Pipeline vs Ground Truth

The standard MiniAn [`pipeline.ipynb`](../pipeline/pipeline.ipynb) runs on a real
recording, where you can *see* the result but never *grade* it. This notebook closes
that loop: we use [**minisim**](https://github.com/miniscope/minisim) to synthesize a
recording from a known forward model, run the **full MiniAn pipeline** on it, and
compare each stage against the **ground truth**. minisim's own notebooks teach the
forward model (`01_anatomy`), demixing (`02_demixing`), and scoring (`03_metrics`);
this is the capstone - a real pipeline scored on known truth.

New to MiniAn? Walk through [`pipeline.ipynb`](../pipeline/pipeline.ipynb) first - it
teaches how to *drive* the pipeline on your own data. This notebook assumes that and asks
the next question: *how well does each stage actually work?*

The pipeline's three relationships to the truth:

| tier | stages | what we do |
|---|---|---|
| **SCORED** | motion, footprints `A`, traces `C`, activity `S` | a real metric vs ground truth |
| **INSPECTED** | glow / background removal, background `b` | no single target; show side-by-side (+ a difference map where a GT image exists) |
| **FORWARD-ONLY** | bleaching, leakage, vasculature | MiniAn has no inverse; they degrade the input |

CNMF refines an initial guess over two spatial+temporal passes. We score the recovery
**after every step** and watch it evolve. The last cell makes the honest point: on real
data you never get this scorecard, so we end with what to trust instead.

In [ ]:
import os, warnings
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MINIAN_INTERMEDIATE"] = "./minian_intermediate"
warnings.filterwarnings("ignore")

import dataclasses as dc
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import holoviews as hv
import ipywidgets as ipw
import mediapy
%matplotlib inline
hv.extension("bokeh")      # bokeh plots stay zoomable/pannable (drag, wheel, box-zoom)

def as2d(x):
    x = np.asarray(x)
    while x.ndim > 2:
        x = x.mean(0) if x.shape[0] != x.shape[-1] else x.sum(0)
    return x

def znorm(x):
    x = np.asarray(x, float)
    return (x - x.mean()) / (x.std() + 1e-9)

def norm01(x):
    x = np.asarray(x, float)
    return x / (np.abs(x).max() + 1e-9)

def play_movie(movie, fps, title=None, max_seconds=12.0):
    # Inline HTML5 player the way minisim's notebooks show recordings (mediapy, ffmpeg
    # under the hood). The inline clip is capped so the embedded video stays small; the
    # full recording can be much longer - pass max_seconds=None to play all of it.
    arr = np.asarray(movie, dtype=np.float32)
    n = arr.shape[0] if max_seconds is None else min(arr.shape[0], int(round(max_seconds * fps)))
    clip = arr[:n]
    vmax = float(np.percentile(clip, 99.7)) or 1.0
    if title is None:
        title = (f"observed recording - first {n / fps:.0f} s"
                 + (f" of {arr.shape[0] / fps:.0f} s" if n < arr.shape[0] else "")
                 + " (contrast-stretched)")
    mediapy.show_video(np.clip(clip / vmax, 0.0, 1.0), fps=fps, title=title)

def sharpness(img):
    # Variance of the Laplacian, a standard focus/sharpness measure. Motion smears cells
    # across the temporal mean; a sharper mean after correction (higher value) is the win.
    img = np.asarray(img, float)
    lap = (-4 * img + np.roll(img, 1, 0) + np.roll(img, -1, 0)
           + np.roll(img, 1, 1) + np.roll(img, -1, 1))
    return float(lap.var())

def bk_lines(series, title, xlabel="frame", ylabel="", width=900, height=300):
    # Zoomable bokeh line overlay from {label: 1d-array}: wheel/box to zoom, drag to pan.
    # The fix for dense plots - a 6000-frame trace you can actually read by zooming in.
    curves = [hv.Curve((np.arange(len(np.asarray(y))), np.asarray(y)), xlabel, ylabel, label=lab)
              for lab, y in series.items()]
    return hv.Overlay(curves).opts(
        hv.opts.Curve(tools=["hover"], line_width=1.3),
        hv.opts.Overlay(width=width, height=height, title=title,
                        legend_position="top_right", active_tools=["box_zoom"]))

def _iou(a, b, thr=0.3):
    # Overlap of two footprints' supports (binarized at thr of each one's peak).
    a, b = a > a.max() * thr, b > b.max() * thr
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0

def cell_navigator(A_est, C_est, A_true, C_true, min_iou=0.0, title="cell"):
    # Scrub matched (estimate <-> ground-truth) cells: the estimated footprint beside the
    # GT footprint, the estimated trace overlaid on the true one. Driven by ipywidgets +
    # matplotlib (inline), which redraws reliably on every slider change in VS Code / Lab /
    # classic - Bokeh/Panel objects re-displayed into an ipywidgets output don't refresh
    # there. Pairs come from the same Hungarian matcher the scorecard uses, best-trace-
    # correlation first; min_iou=0 keeps every matched pair (rough seeds included), raise it
    # to show only clean cells.
    A_est, A_true = np.asarray(A_est), np.asarray(A_true)
    C_est, C_true = np.asarray(C_est), np.asarray(C_true)
    pairs = [(t, e) for e, t in hungarian_match(A_est, A_true).pairing
             if _iou(A_est[e], A_true[t]) >= min_iou]
    def corr(t, e):
        return float(np.corrcoef(znorm(C_true[t]), znorm(C_est[e]))[0, 1])
    pairs.sort(key=lambda te: -corr(*te))
    if not pairs:
        print(f"{title}: no estimate matched a GT cell at IoU >= {min_iou}.")
        return
    from IPython.display import display, clear_output
    out = ipw.Output()
    def render(rank):
        t, e = pairs[rank]
        with out:
            clear_output(wait=True)   # replace the previous figure in place
            fig, ax = plt.subplots(1, 3, figsize=(14, 3.4), gridspec_kw={"width_ratios": [1, 1, 3]})
            ax[0].imshow(norm01(A_est[e]), cmap="magma"); ax[0].axis("off")
            ax[0].set_title(f"estimate (unit {e})")
            ax[1].imshow(norm01(A_true[t]), cmap="magma"); ax[1].axis("off")
            ax[1].set_title(f"ground truth (cell {t}, IoU {_iou(A_est[e], A_true[t]):.2f})")
            ax[2].plot(znorm(C_true[t]), lw=1.5, label="ground truth")
            ax[2].plot(znorm(C_est[e]), lw=0.9, label="estimate", alpha=0.85)
            ax[2].set_title(f"{title} rank {rank} of {len(pairs) - 1}  ·  trace r = {corr(t, e):.2f}")
            ax[2].set_xlabel("frame"); ax[2].set_yticks([]); ax[2].legend(fontsize=8)
            fig.tight_layout()
            display(fig); plt.close(fig)   # show once, then free it: avoids VS Code duplicate renders
    slider = ipw.IntSlider(min=0, max=len(pairs) - 1, step=1, value=0,
                           description=f"{title} #", continuous_update=False)
    slider.observe(lambda change: render(change["new"]), names="value")
    render(0)
    display(ipw.VBox([slider, out]))

def triptych(result, truth, label_result, label_truth):
    # minian result | ground truth | normalized difference map, with a scalar so the
    # comparison is graded, not just eyeballed: spatial correlation and relative residual.
    r, t = norm01(result), norm01(truth)
    rf, tf = r.ravel(), t.ravel()
    corr = float(np.corrcoef(rf, tf)[0, 1]) if rf.std() > 1e-9 and tf.std() > 1e-9 else float("nan")
    resid = float(np.sqrt(((r - t) ** 2).mean()) / (np.sqrt((t ** 2).mean()) + 1e-9))
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    ax[0].imshow(r, cmap="magma"); ax[0].set_title(label_result)
    ax[1].imshow(t, cmap="magma"); ax[1].set_title(label_truth)
    im = ax[2].imshow(r - t, cmap="RdBu_r", vmin=-1, vmax=1)
    ax[2].set_title(f"difference  (corr={corr:.2f}, residual={resid:.2f})")
    for a in ax: a.axis("off")
    fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
    fig.tight_layout(); plt.show()

## 1. Build a synthetic recording

`quick` is a small, lightly-overlapping field carrying the full forward model (diffuse
neuropil, illumination + vignette shading and a leakage floor - the "glow" - and an
absorbing vascular tree) that the pipeline recovers cleanly. `ca1` / `cortex_l23` are
realistic, dense fields built from minisim's tissue presets; they are deliberately
harder and the default parameters below recover them only partially - that gap is a
lesson in itself.

In [ ]:
from minisim import presets, simulate
from minisim.testing import make_recording, score, Estimate
from minisim.spec import (Neuropil, Vignette, IlluminationProfile, Leakage,
                          Bleaching, BrainMotion, Vasculature, VesselLayer)

# `quick` is built with make_recording, a bare builder, so we add the forward-model
# effects (neuropil, illumination/vignette shading, a leakage floor, a vascular tree)
# to make them visibly present. The region presets are full microscope + tissue models
# that already carry that structure, so they get nothing injected (see build_dataset).
EFFECTS = [Neuropil(amplitude=2.0, n_components=4),
           IlluminationProfile(falloff=0.88), Vignette(falloff=0.65),
           Leakage(level=0.5), Bleaching()]
QUICK_VASC = Vasculature(enabled=True, layers=[VesselLayer(
    depth_um=22.0, n_roots=4, root_radius_um=22.0, min_radius_um=3.0,
    opacity=0.9, branch_prob=0.3)])

def build_dataset(name="quick", duration_s=50.0, fps=20.0):
    if name == "quick":
        return make_recording(n_cells=40, n_px=160, duration_s=duration_s, fps=fps,
                              seed=1, motion=True, extra_steps=EFFECTS + [QUICK_VASC],
                              save_intermediates=True)
    # Region presets are full microscope + tissue models: the V4 scope carries the optics,
    # illumination, vignette, leakage and exposure; the tissue preset adds neuropil and
    # vasculature. We inject nothing - only downscale the sensor to 256 px to keep the demo
    # tractable - and add BrainMotion so the motion stage has a real shift to recover.
    base = presets.miniscope_v4()
    scope = base.model_copy(update={"image_sensor": base.image_sensor.model_copy(
        update={"n_px_height": 256, "n_px_width": 256})})
    region = getattr(presets, name)()
    return simulate(presets.build_spec(scope, region, duration_s=duration_s, fps=fps,
                                       extra_steps=[BrainMotion()],
                                       save_intermediates=True))

DATASET = "quick"   # <- try "ca1" or "cortex_l23"
rec = build_dataset(DATASET)
gt = rec.ground_truth
print(f"{DATASET}: {gt.n_units} cells planted, "
      f"{int(np.asarray(gt.detectable).sum())} detectable, "
      f"{rec.observed_movie.sizes['frame']} frames, "
      f"{rec.observed_movie.sizes['height']}x{rec.observed_movie.sizes['width']} px")

### Watch the recording

Before decomposing it, play the movie MiniAn will actually see - the same inline player
minisim's notebooks use. This is the raw input: shading, glow, vasculature, brain motion
and all. A short clip is embedded; set `max_seconds=None` to play the whole thing.

In [ ]:
# `interactive` gates the interactive-only cells (this video player and the cell
# navigators below), mirroring pipeline.ipynb. The test suite sets MINIAN_INTERACTIVE=False
# so batch runs and CI skip them; leave it unset for the full interactive experience.
interactive = os.getenv("MINIAN_INTERACTIVE", "True").lower() == "true"
if interactive:
    fps = float(rec.spec.acquisition.fps)
    play_movie(rec.observed_movie, fps)

### The forward model, stage by stage

minisim composes physical effects; with `save_intermediates` each leaves a snapshot -
this is the "truth" the pipeline will try to undo.

In [ ]:
snaps = list(rec.snapshots.items())
ncol = max(1, (len(snaps) + 1) // 2)
fig, axes = plt.subplots(2, ncol, figsize=(3 * ncol, 6))
for ax, (name, snap) in zip(axes.ravel(), snaps):
    ax.imshow(as2d(snap), cmap="magma"); ax.set_title(name, fontsize=9); ax.axis("off")
for ax in axes.ravel()[len(snaps):]:
    ax.axis("off")
fig.suptitle(f"Forward-model snapshots ({DATASET})"); fig.tight_layout(); plt.show()

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4))
a0.imshow(np.asarray(gt.A_observed).max(0), cmap="magma")
a0.set_title("True footprints  A_observed.max(cell)"); a0.axis("off")
Ctrue = np.asarray(gt.C)
for i in range(min(6, Ctrue.shape[0])):
    a1.plot(Ctrue[i] / (Ctrue[i].max() + 1e-9) + i, lw=0.8)
a1.set_title("True calcium traces (6 cells)"); a1.set_xlabel("frame"); a1.set_yticks([])
fig.tight_layout(); plt.show()

## Parameters

As in `pipeline.ipynb`, every stage is driven by a parameter dict set up front. Good
values depend on the data (frame rate, SNR, cell size); these mirror the pipeline
notebook's defaults and work for `quick`. Dense presets generally need re-tuning.

In [ ]:
param_seeds_init   = {"wnd_size": 1000, "method": "rolling", "stp_size": 500, "max_wnd": 15, "diff_thres": 3}
param_pnr_refine   = {"noise_freq": 0.06, "thres": 1}
param_ks_refine    = {"sig": 0.05}
param_seeds_merge  = {"thres_dist": 10, "thres_corr": 0.8, "noise_freq": 0.06, "chunk": 600}
param_initialize   = {"thres_corr": 0.8, "wnd": 10, "noise_freq": 0.06, "chunk": 600}
param_init_merge   = {"thres_corr": 0.8, "chunk": 600}
param_get_noise    = {"noise_range": (0.06, 0.5)}
param_first_spatial   = {"dl_wnd": 10, "sparse_penal": 0.01, "size_thres": (25, None)}
param_first_temporal  = {"noise_freq": 0.06, "sparse_penal": 1, "p": 1, "add_lag": 20, "jac_thres": 0.2}
param_first_merge     = {"thres_corr": 0.8, "chunk": 600}
param_second_spatial  = {"dl_wnd": 10, "sparse_penal": 0.01, "size_thres": (25, None)}
param_second_temporal = {"noise_freq": 0.06, "sparse_penal": 1, "p": 1, "add_lag": 20, "jac_thres": 0.4}

In [ ]:
from dask.distributed import Client, LocalCluster, wait

from minian.cnmf import (compute_trace, get_noise_fft, unit_merge,
                         update_spatial, update_temporal, update_background)
from minian.initialization import (initA, initC, ks_refine, pnr_refine,
                                   seeds_init, seeds_merge)
from minian.motion_correction import apply_transform, estimate_motion
from minian.preprocessing import denoise, remove_background
from minian.utilities import get_optimal_chk
from minisim.metrics import hungarian_match, shift_rmse

# The synthetic recording is tiny, so run an in-process threaded cluster
# (processes=False, memory monitor off): no worker subprocesses and no distributed
# memory-pause stalls. A process-based cluster - even matching pipeline.ipynb's
# memory-throttled setup - hangs under nbconvert on the constrained macOS CI runner;
# threads sidestep that entirely and are plenty for this data. pipeline.ipynb keeps the
# process-based, memory-throttled cluster for full-size recordings.
cluster = LocalCluster(processes=False, threads_per_worker=4, memory_limit=0,
                       dashboard_address=":0")
client = Client(cluster)

# 8-bit sensor: feed uint8 (cv2.medianBlur in denoise needs CV_8U).
varr = rec.observed_movie.astype(np.uint8).assign_coords(
    frame=np.arange(rec.observed_movie.sizes["frame"]),
    height=np.arange(rec.observed_movie.sizes["height"]),
    width=np.arange(rec.observed_movie.sizes["width"]))
chk, _ = get_optimal_chk(varr, dtype=float)
varr = varr.chunk({"frame": chk["frame"], "height": -1, "width": -1})
client

## 2. Glow removal  ·  *INSPECTED*

MiniAn subtracts the per-pixel minimum ("glow"). That subtracted floor *is* an image, so
we show it: the min projection MiniAn removes, lined up against the GT shading
(illumination x vignette) that physically caused it. There is no single ground-truth glow
image - the floor also blends in a leakage offset and out-of-focus light, which minisim
tracks separately - so we don't score it; we show what was taken out and compare.

In [ ]:
glow = varr.min("frame").compute().values        # the per-pixel floor MiniAn subtracts
varr_ref = (varr - varr.min("frame")).persist(); wait(varr_ref)
# GT shading (illumination x vignette) is only tracked separately when injected as
# steps; presets that bake it into the optics leave these None, so show the GT panel
# only when it exists.
shade = None
if getattr(gt, "illumination", None) is not None:
    shade = as2d(gt.illumination)
if getattr(gt, "vignette", None) is not None:
    shade = as2d(gt.vignette) if shade is None else shade * as2d(gt.vignette)

panels = [("observed (mean)", varr.mean("frame")),
          ("removed glow (min projection)", glow),
          ("after glow removal", varr_ref.mean("frame").compute())]
if shade is not None:
    panels.append(("GT shading: illumination x vignette", shade))
    corr = float(np.corrcoef(norm01(glow).ravel(), norm01(shade).ravel())[0, 1])
    print(f"removed glow vs GT shading: spatial corr = {corr:.2f}")
fig, ax = plt.subplots(1, len(panels), figsize=(4.0 * len(panels), 4))
for a, (t, img) in zip(np.atleast_1d(ax), panels):
    a.imshow(np.asarray(img), cmap="magma"); a.set_title(t, fontsize=9); a.axis("off")
fig.suptitle("Glow removal: the floor MiniAn subtracts vs the GT shading that caused it")
fig.tight_layout(); plt.show()

## 3. Background removal  ·  *INSPECTED + difference map*

A median denoise + morphological tophat removes diffuse background. The ideal result is
the cells alone - minisim's `cells_only` snapshot - so here we *can* line them up and
look at the residual difference (the background the heuristic failed to remove, or the
cell signal it ate).

In [ ]:
varr_ref = denoise(varr_ref, **{"method": "median", "ksize": 7})
varr_ref = remove_background(varr_ref, **{"method": "tophat", "wnd": 15}).persist(); wait(varr_ref)
after_bg = varr_ref.mean("frame").compute().values

cells_only = np.asarray(rec.snapshots["cells_only"]).mean(0)
fo, fs = getattr(gt, "fov_offset", None), getattr(gt, "fov_shape", None)
if fo is not None and fs is not None and cells_only.shape != after_bg.shape:
    cells_only = cells_only[fo[0]:fo[0] + fs[0], fo[1]:fo[1] + fs[1]]
triptych(after_bg, cells_only, "after background removal", "GT cells-only (ideal)")

## 4. Motion correction  ·  *SCORED*

Truth is exact: minisim applied a known per-frame shift. We estimate it and score the
recovered trajectory against `ground_truth.shifts` (as a *correction*, origin aligned).
The shift curves are **interactive** (bokeh): drag to pan, scroll or box-select to zoom -
the only way to read a recording that runs to thousands of frames.

In [ ]:
motion = estimate_motion(varr_ref, **{"dim": "frame"}).persist(); wait(motion)
Y = apply_transform(varr_ref, motion, fill=0)
Y_fm = Y.astype(float).chunk({"frame": chk["frame"], "height": -1, "width": -1}).persist()
Y_hw = Y_fm.chunk({"frame": -1, "height": chk["height"], "width": chk["width"]}).persist()
wait([Y_fm, Y_hw]); max_proj = Y_fm.max("frame").compute()

est = motion.transpose("frame", "shift_dim").values
true = np.asarray(gt.shifts)
rmse = shift_rmse(est, true, correction=True, align=True)
print(f"Motion RMSE = {rmse:.3f} px over {true.shape[0]} frames")
# overlay true vs estimated (origin-aligned correction) for each axis; zoomable
est_corr = {lab: -(est[:, k] - est[:, k].mean()) + true[:, k].mean() for k, lab in enumerate(["dy", "dx"])}
(bk_lines({"true shift": true[:, 0], "estimated (corr.)": est_corr["dy"]},
          f"dy  (RMSE {rmse:.3f} px)", ylabel="pixels", height=240)
 + bk_lines({"true shift": true[:, 1], "estimated (corr.)": est_corr["dx"]},
            "dx", ylabel="pixels", height=240)).cols(1)

Motion smears every cell across the temporal mean. The clearest proof that correction
worked is the **mean projection**: blurred before, crisp after, lined up against the
motion-free ground truth. We quantify it with a sharpness score (variance of the
Laplacian) - it should jump from *before* to *after*.

In [ ]:
before_mean = varr_ref.mean("frame").compute().values   # motion still present
after_mean = Y_fm.mean("frame").compute().values        # motion removed

cells_only_m = np.asarray(rec.snapshots["cells_only"]).mean(0)   # motion-free truth
fo, fs = getattr(gt, "fov_offset", None), getattr(gt, "fov_shape", None)
if fo is not None and fs is not None and cells_only_m.shape != after_mean.shape:
    cells_only_m = cells_only_m[fo[0]:fo[0] + fs[0], fo[1]:fo[1] + fs[1]]

s_before, s_after = sharpness(before_mean), sharpness(after_mean)
print(f"mean-projection sharpness: before={s_before:.3g}  after={s_after:.3g}  "
      f"({s_after / (s_before + 1e-12):.2f}x)")
b01, a01, g01 = norm01(before_mean), norm01(after_mean), norm01(cells_only_m)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(b01, cmap="magma"); ax[0].set_title(f"before MC (sharpness {s_before:.3g})")
ax[1].imshow(a01, cmap="magma"); ax[1].set_title(f"after MC (sharpness {s_after:.3g})")
ax[2].imshow(g01, cmap="magma"); ax[2].set_title("GT cells-only (motion-free)")
im = ax[3].imshow(a01 - g01, cmap="RdBu_r", vmin=-1, vmax=1)
ax[3].set_title("difference  (after MC - GT)")
for a in ax: a.axis("off")
fig.colorbar(im, ax=ax[3], fraction=0.046, pad=0.04)
fig.suptitle("Motion correction sharpens the mean projection toward truth")
fig.tight_layout(); plt.show()

## 5. Initialization - the first guess  ·  *SCORED*

Detect seeds, refine (PNR, KS) and merge them, then build an initial spatial matrix `A`
and initial traces `C`. This is the pipeline's first guess - we score it against truth
and will watch CNMF refine it.

In [ ]:
seeds = seeds_init(Y_fm, **param_seeds_init)
seeds, pnr, gmm = pnr_refine(Y_hw, seeds, **param_pnr_refine)
seeds = ks_refine(Y_hw, seeds, **param_ks_refine)
seeds_final = seeds[seeds["mask_ks"] & seeds["mask_pnr"]].reset_index(drop=True)
seeds_final = seeds_merge(Y_hw, max_proj, seeds_final, **param_seeds_merge)

A = initA(Y_hw, seeds_final[seeds_final["mask_mrg"]], **param_initialize).persist()
C = initC(Y_fm, A).persist()
try:
    A, C = unit_merge(A, C, **param_init_merge)
except ValueError:
    pass  # no overlapping units to merge
A, C = A.persist(), C.persist()
C_chk = C.chunk({"unit_id": -1, "frame": chk["frame"]}).persist()
b, f = update_background(Y_fm, A, C_chk); b, f = b.persist(), f.persist()
sn = get_noise_fft(Y_hw, **param_get_noise).persist()
wait([A, C, b, f, sn])

# --- per-stage scorecard, accumulated so we can watch the recovery evolve ---
history = []
def record(stage, A, C, S=None):
    # update_temporal can return C/S with their units REORDERED relative to A (same
    # set, different order). Align by unit_id before converting to numpy, or each
    # footprint gets paired with the wrong trace - silent on quick, fatal on dense.
    C = C.sel(unit_id=A.coords["unit_id"])
    if S is not None:
        S = S.sel(unit_id=A.coords["unit_id"])
    Anp = np.asarray(A.transpose("unit_id", "height", "width"))
    Cnp = np.asarray(C.transpose("unit_id", "frame"))
    Snp = np.asarray(S.transpose("unit_id", "frame")) if S is not None else None
    r = score(Estimate(A=Anp, C=Cnp, S=Snp, shifts=est), gt)
    history.append({"stage": stage, "report": r, "A": Anp, "C": Cnp})
    print(f"{stage:11s} recall={r.recall:.2f}  precision={r.precision:.2f}  "
          f"IoU={r.mean_overlap:.2f}  trace_r={r.trace_corr:.2f}")
    return r

record("init", A, C)

In [ ]:
# the initial guess vs ground truth
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(np.asarray(gt.A_observed).max(0), cmap="magma"); ax[0].set_title("true footprints")
ax[1].imshow(history[-1]["A"].max(0), cmap="magma"); ax[1].set_title("initial A guess")
for a in ax: a.axis("off")
fig.tight_layout(); plt.show()

### The initial seeds vs ground truth  ·  scrub through them

Before CNMF touches them, each seed already has a footprint (`initA`) and a rough trace
(`initC`, the mean of its footprint pixels over time). Drag the slider to step through the
seeds: each seeded footprint sits beside the GT cell it best matches, and its trace is laid
over the true one. This is the *raw first guess* - footprints are blobby and the traces
noisy, but the events are already there. Every match is shown (best trace first), so the
rough ones at the end are honest too; watch how much the CNMF passes below tighten both.

In [ ]:
if interactive:
    cell_navigator(history[-1]["A"], history[-1]["C"], gt.A_observed, gt.C,
                   min_iou=0.0, title="seed")

## 6. CNMF pass 1 - spatial update  ·  *SCORED*

Re-solve each footprint given the current traces. Footprints tighten toward the true
cell shapes (watch IoU rise).

**Parameter exploration (scored).** `pipeline.ipynb` tunes parameters by changing a value
and *looking* at the result. Here we can do better: sweep the spatial `sparse_penal` and
**grade each choice against truth**. Too small and footprints bleed into their neighbors
(IoU falls); too large and real cells are eroded or dropped (recall falls). The pipeline
uses a value in the good range - and on real data, where there is no score, this is the
intuition you carry away: the setting that yields compact, cell-shaped footprints.

In [ ]:
penalties = [0.001, 0.01, 0.1, 1.0]
sweep = []
for sp_pen in penalties:
    A_t, mask_t, nf_t = update_spatial(Y_hw, A, C, sn, in_memory=True,
                                       dl_wnd=10, sparse_penal=sp_pen, size_thres=(25, None))
    A_t = A_t.persist(); wait(A_t)
    C_t = C.sel(unit_id=mask_t) * nf_t
    r = score(Estimate(A=np.asarray(A_t.transpose("unit_id", "height", "width")),
                       C=np.asarray(C_t.transpose("unit_id", "frame")), shifts=est), gt)
    sweep.append((sp_pen, r.recall, r.mean_overlap))
    print(f"sparse_penal={sp_pen:<6} recall={r.recall:.2f}  IoU={r.mean_overlap:.2f}  "
          f"units={int(A_t.sizes['unit_id'])}")

fig, ax = plt.subplots(figsize=(7, 3.5))
xs = [s[0] for s in sweep]
ax.plot(xs, [s[1] for s in sweep], "o-", label="recall")
ax.plot(xs, [s[2] for s in sweep], "s-", label="footprint IoU")
ax.axvline(param_first_spatial["sparse_penal"], color="0.6", ls="--", lw=1, label="value used")
ax.set_xscale("log"); ax.set_xlabel("spatial sparse_penal"); ax.set_ylabel("score vs truth")
ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
ax.set_title("Tuning spatial sparsity against ground truth"); fig.tight_layout(); plt.show()

In [ ]:
A_new, mask, norm_fac = update_spatial(Y_hw, A, C, sn, in_memory=True, **param_first_spatial)
A = A_new.persist(); C = (C.sel(unit_id=mask) * norm_fac).persist()
C_chk = C.chunk({"unit_id": -1, "frame": chk["frame"]}).persist()
b, f = update_background(Y_fm, A, C_chk); b, f = b.persist(), f.persist()
wait([A, C, b, f])
record("spatial 1", A, C)

## 7. CNMF pass 1 - temporal update  ·  *SCORED*

Extract each cell's trace (`compute_trace`) and deconvolve it into a denoised calcium
trace `C` and activity `S`, then merge units that turned out to be the same cell.

In [ ]:
YrA = compute_trace(Y_fm, A, b, C_chk, f).chunk({"unit_id": 1, "frame": -1}).persist()
C_new, S_new, b0, c0, g, mask = update_temporal(A, C, YrA=YrA, **param_first_temporal)
# update_temporal reorders units; realign A to C_new's order so the next merge and
# spatial pass pair each footprint with its own trace (misalignment corrupts pass 2).
A = A.sel(unit_id=C_new.coords["unit_id"]).persist(); C = C_new.persist(); S = S_new.persist()
wait([A, C, S])
record("temporal 1", A, C, S)

try:
    A, C = unit_merge(A, C, **param_first_merge)
    A, C = A.persist(), C.persist()
except ValueError:
    pass
C_chk = C.chunk({"unit_id": -1, "frame": chk["frame"]}).persist()
b, f = update_background(Y_fm, A, C_chk); b, f = b.persist(), f.persist(); wait([A, C, b, f])

## 8. CNMF pass 2 - spatial then temporal  ·  *SCORED*

A second pass, exactly as `pipeline.ipynb` does, refines footprints and traces again.

In [ ]:
A_new, mask, norm_fac = update_spatial(Y_hw, A, C, sn, in_memory=True, **param_second_spatial)
A = A_new.persist(); C = (C.sel(unit_id=mask) * norm_fac).persist()
C_chk = C.chunk({"unit_id": -1, "frame": chk["frame"]}).persist()
b, f = update_background(Y_fm, A, C_chk); b, f = b.persist(), f.persist()
wait([A, C, b, f])
record("spatial 2", A, C)

YrA = compute_trace(Y_fm, A, b, C_chk, f).chunk({"unit_id": 1, "frame": -1}).persist()
C_new, S_new, b0, c0, g, mask = update_temporal(A, C, YrA=YrA, **param_second_temporal)
# update_temporal reorders units; realign A to C_new's order so the next merge and
# spatial pass pair each footprint with its own trace (misalignment corrupts pass 2).
A = A.sel(unit_id=C_new.coords["unit_id"]).persist(); C = C_new.persist(); S = S_new.persist()
wait([A, C, S])
record("temporal 2", A, C, S)

## 9. Watch it improve

Because every CNMF step was scored against the truth, we can *watch* recovery evolve
instead of guessing. On a **clean** field (`quick`) the initial guess is already strong,
so CNMF mainly sharpens footprints; on a **dense** field (`ca1`/`cortex_l23`) you see the
real climb - each pass recovers more cells and tightens their shapes. The footprint
montage and one matched cell's trace converging on its true signal tell the same story.

In [ ]:
stages = [h["stage"] for h in history]
fig, ax = plt.subplots(figsize=(9, 3.5))
for key, lab in [("recall", "recall"), ("mean_overlap", "footprint IoU"),
                 ("trace_corr", "trace r")]:
    ax.plot(stages, [getattr(h["report"], key) for h in history], marker="o", label=lab)
ax.set_ylim(0, 1.05); ax.set_ylabel("score"); ax.legend()
ax.set_title("Recovery vs CNMF stage"); fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(history) + 1, figsize=(2.6 * (len(history) + 1), 3))
axes[0].imshow(np.asarray(gt.A_observed).max(0), cmap="magma")
axes[0].set_title("ground truth", fontsize=9)
for ax, h in zip(axes[1:], history):
    ax.imshow(h["A"].max(0), cmap="magma"); ax.set_title(h["stage"], fontsize=9)
for ax in axes: ax.axis("off")
fig.suptitle("Footprints sharpening across CNMF"); fig.tight_layout(); plt.show()

In [ ]:
# one matched cell's trace converging on truth across stages (zoomable)
final_match = hungarian_match(history[-1]["A"], np.asarray(gt.A_observed))
gt2est_final = {t: e for e, t in final_match.pairing}
g = next(iter(gt2est_final), None)
if g is not None:
    series = {"true": znorm(np.asarray(gt.C)[g])}
    for h in history:
        g2e = {t: e for e, t in hungarian_match(h["A"], np.asarray(gt.A_observed)).pairing}
        if g in g2e:
            series[h["stage"]] = znorm(h["C"][g2e[g]])
    conv = bk_lines(series, f"Cell {g}: recovered trace converging on truth (z-scored)",
                    ylabel="z-score", height=320)
else:
    conv = None
conv

### Which cells were found, and which were missed

CNMF is not built to find every cell - it recovers the ones it can resolve and lets the
rest go. With ground truth we can show exactly which detectable cells were recovered and
which were missed, and confirm that the units it *did* keep are real (high precision),
not spurious. On `quick` almost everything is found; on a dense field a real fraction is
honestly missed - the overlapping and dim cells - and that is expected, not a failure.

In [ ]:
A_fin = history[-1]["A"]; Atrue = np.asarray(gt.A_observed)
match = hungarian_match(A_fin, Atrue)

THR = 0.3   # footprint IoU a match must clear to count as "recovered" (see _iou helper)
matched_e = {}              # true cell index -> estimated unit index (recovered)
for e, t in match.pairing:
    if _iou(A_fin[e], Atrue[t]) >= THR:
        matched_e[t] = e
rec_t = set(matched_e)
missed_t = set(range(Atrue.shape[0])) - rec_t
tp_e = set(matched_e.values())

def _cent(F):
    ys, xs = np.nonzero(F > F.max() * 0.3)
    return (np.nan, np.nan) if not len(xs) else (xs.mean(), ys.mean())

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(Atrue.max(0), cmap="gray")
for t in rec_t:
    x, y = _cent(Atrue[t]); ax[0].plot(x, y, "o", mfc="none", mec="lime", ms=6)
for t in missed_t:
    x, y = _cent(Atrue[t]); ax[0].plot(x, y, "x", color="red", ms=5)
ax[0].set_title(f"GT cells: recovered {len(rec_t)} (green)  ·  missed {len(missed_t)} (red x)")
ax[1].imshow(A_fin.max(0), cmap="gray")
for e in range(A_fin.shape[0]):
    x, y = _cent(A_fin[e])
    ax[1].plot(x, y, "o", mfc="none", mec="lime" if e in tp_e else "magenta", ms=6)
ax[1].set_title("Estimated units: matched (green)  ·  unmatched / false (magenta)")
for a in ax: a.axis("off")
fig.suptitle(f"Detection (IoU>={THR}): CNMF keeps real cells; misses overlapping/dim ones")
fig.tight_layout(); plt.show()

### Navigate every matched cell  ·  footprint and trace vs truth

The scorecard is an average; this is the per-cell reality behind it. Drag the slider to
step through every recovered cell (best trace match first) and see its footprint beside
the true one and its trace overlaid on the truth. This is where you build the eye for what
"recovered well" actually looks like - and spot the ones the average is hiding.

In [ ]:
# same navigator as the seeds, now over the FINAL recovery, restricted to cells that
# actually cleared the IoU threshold (the recovered set, not every rough match)
if interactive:
    cell_navigator(history[-1]["A"], history[-1]["C"], gt.A_observed, gt.C,
                   min_iou=THR, title="recovered")

### Demixing: separating overlapping cells

This is the job CNMF exists for. Where two cells overlap, their fluorescence is *mixed*
in every shared pixel; CNMF factors that mixture back into one trace per cell. We find a
recovered, overlapping pair, compare the raw mixed-pixel signal to the two demixed traces,
and overlay each on its truth. (A well-separated field has no such pair - then demixing is
trivial and this cell says so.)

In [ ]:
Ctrue = np.asarray(gt.C); Cfin = history[-1]["C"]
rec_list = sorted(rec_t)
pair = None
for i in range(len(rec_list)):
    for j in range(i + 1, len(rec_list)):
        if _iou(Atrue[rec_list[i]], Atrue[rec_list[j]]) > 0.05:
            pair = (rec_list[i], rec_list[j]); break
    if pair: break

if pair is None:
    print("No overlapping recovered pair - this field is well separated, so demixing is trivial here.")
else:
    t1, t2 = pair; e1, e2 = matched_e[t1], matched_e[t2]
    union = (Atrue[t1] > Atrue[t1].max() * 0.3) | (Atrue[t2] > Atrue[t2].max() * 0.3)
    ys, xs = np.nonzero(union)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    sub = np.asarray(Y_fm.isel(height=slice(y0, y1), width=slice(x0, x1)).transpose("frame", "height", "width"))
    mixed = sub[:, union[y0:y1, x0:x1]].mean(1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 3]})
    ov = np.zeros(Atrue[t1].shape + (3,))
    ov[..., 0] = norm01(Atrue[t1]); ov[..., 1] = norm01(Atrue[t2])
    ax[0].imshow(ov[y0:y1, x0:x1]); ax[0].set_title("overlapping GT pair\\n(cell 1 red, cell 2 green)"); ax[0].axis("off")
    ax[1].plot(znorm(mixed), color="0.6", lw=1.0, label="raw mixed pixels")
    ax[1].plot(znorm(Ctrue[t1]) + 5, "r", lw=1.4, label="cell 1 true")
    ax[1].plot(znorm(Cfin[e1]) + 5, "k", lw=0.8, alpha=0.8, label="cell 1 demixed")
    ax[1].plot(znorm(Ctrue[t2]) + 10, "g", lw=1.4, label="cell 2 true")
    ax[1].plot(znorm(Cfin[e2]) + 10, "k", lw=0.8, alpha=0.8, label="cell 2 demixed")
    ax[1].set_title("the mixed signal, demixed into one faithful trace per cell")
    ax[1].set_xlabel("frame"); ax[1].set_yticks([]); ax[1].legend(fontsize=7, ncol=3)
    fig.tight_layout(); plt.show()

## 10. The background component `b`  ·  *INSPECTED*

CNMF closes with a low-rank background term, `b` (spatial) times `f` (temporal), that
absorbs whatever diffuse fluorescence the cell model `A·C` left behind. It is tempting to
read `b` as a picture of the neuropil, but it is not: `b` is a *residual*, the time-mean of
`clip(Y - A·C, 0)`. Two things shape it, and neither is neuropil:

- The physical diffuse background in this recording is dominated by **shading**
  (illumination x vignette + the leakage glow), and that was already removed upstream in
  steps 2-3 - by the time CNMF runs there is little diffuse signal left. The injected
  neuropil is faint here (about 7% of what preprocessing removes, and barely above the
  noise in the raw movie), so there is no clean neuropil for `b` to recover.
- What remains in `b` is mostly **per-cell subtraction residue**: rings where `A` does not
  perfectly match a footprint, plus a positive bias from the `clip(0)` - clamping negative
  residuals to zero pushes the time-mean up exactly where signal variance is high, i.e. on
  the cells.

So expect `b` to look like faint cell-shaped blobs on a near-flat field, not a smooth
neuropil map. That is the lesson: `b` is a mathematical mop-up term, not a physical
estimate of any one source - don't read anatomy into it.

In [ ]:
bnp = np.asarray(b.transpose("height", "width")) if "height" in b.dims else as2d(b)
fig, a = plt.subplots(figsize=(4.8, 4.2))
im = a.imshow(bnp, cmap="magma"); a.axis("off")
a.set_title("MiniAn background b  (CNMF residual)")
fig.colorbar(im, ax=a, fraction=0.046, pad=0.04)
fig.tight_layout(); plt.show()
bk_lines({"f": np.asarray(f).ravel()}, "MiniAn background temporal f",
         ylabel="intensity", height=240)

## 11. The scorecard

One call grades the whole recovery against ground truth. Each bar is in [0, 1], higher is
better:

- **recall** - fraction of the *detectable* GT cells that MiniAn recovered
  (`n_matched / n_true`). The denominator is the detectable subset, not every planted
  cell, so cells too dim for any method to find are not held against it.
- **precision** - fraction of MiniAn's components that match a real cell
  (`n_matched / n_est`). Low precision means spurious or duplicate footprints.
- **f1** - harmonic mean of recall and precision: one detection number that punishes a
  lopsided result (high recall bought with junk units, or vice versa).
- **IoU** - mean footprint overlap (intersection-over-union) over matched pairs: how well
  the recovered footprint *shape* lines up with truth.
- **trace r** - median Pearson correlation between each matched cell's recovered
  fluorescence trace `C` and its true trace: temporal fidelity.
- **activity r** - median Pearson correlation between the deconvolved activity `S` and the
  true spiking: how well event timing and relative amplitude are recovered (up to an
  unidentifiable gain).

`report.summary()` (printed below the chart) adds the population counts (detectable of
planted, and the recall denominator used) and, when motion is present, the **motion RMSE**
in pixels.

In [ ]:
uid = C.coords["unit_id"]   # align footprints/spikes to the (reordered) traces
report = score(Estimate(A=A.sel(unit_id=uid).transpose("unit_id", "height", "width").values,
                        C=C.transpose("unit_id", "frame").values,
                        S=S.sel(unit_id=uid).transpose("unit_id", "frame").values, shifts=est), gt)
print(report.summary())

vals = {"recall": report.recall, "precision": report.precision, "f1": report.f1,
        "IoU": report.mean_overlap, "trace r": report.trace_corr,
        "activity r": report.activity_corr}
vals = {k: (0 if v != v else v) for k, v in vals.items()}
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(list(vals), list(vals.values()), color="teal"); ax.set_ylim(0, 1)
ax.set_title(f"Recovery scorecard ({DATASET})")
for i, v in enumerate(vals.values()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)
fig.tight_layout(); plt.show()
client.close(); cluster.close()

## 12. What to trust when there is no scorecard

On a real recording you will never have `ground_truth` - the loop you just closed can't
be closed again. Use this notebook to build intuition for what each stage *should* look
like, so on real data you can lean on the proxies it taught:

- **Motion**: a sharper temporal-mean after correction.
- **Footprints**: compact, cell-shaped, non-overlapping `A` that *tightened* across CNMF
  passes; blobs that grow or merge are a warning.
- **Traces**: calcium-like rises with slow decays; deconvolved `S` sparse.
- **Caveat**: `S` is not a spike train and trace scale is not identifiable - compare
  shape and timing, never absolute amplitude.

Switch `DATASET` to `ca1` or `cortex_l23` to see a dense, realistic field: MiniAn recovers
a high-precision subset and honestly misses the most crowded and dim cells. That is not a
bug - CNMF is a demixer, not an oracle - and the parameters that nailed `quick` need
re-tuning here (see minisim's `02_demixing`).

### Score your own parameter choices

The real payoff: before touching a real recording, simulate one shaped like *your* rig,
run MiniAn with *your* parameters, and grade them - the same `score()` call used throughout.

```python
from minisim import presets, simulate
from minisim.testing import score, Estimate
# a recording shaped like your experiment (scope, region, frame rate, duration):
rec = simulate(presets.build_spec(presets.miniscope_v4(), presets.ca1()))
# ...run the MiniAn pipeline on rec.observed_movie to get A, C, S, shifts...
report = score(Estimate(A=A, C=C, S=S, shifts=shifts), rec.ground_truth)
print(report.summary())   # recall / precision / IoU / trace & activity correlation
```